In [144]:
import cv2
import mediapipe as mp
import numpy as np
import os
import tensorflow as tf
# import pyttsx3
# import threading

In [145]:
# speech_engine = pyttsx3.init()
# speech_engine.setProperty("rate", 150)
# speech_engine.setProperty("volume", 1.0)
# last_spoken_message = ""

In [146]:
# def speak_message(message):
#     global last_spoken_message
#     if message != last_spoken_message:
#         last_spoken_message = message
#         def speak():
#             speech_engine.say(message)
#             speech_engine.runAndWait()
#         threading.Thread(target=speak, daemon=True).start()

In [147]:
project_folder='sign_free_project'
data_folder=os.path.join(project_folder,'data')
model_folder=os.path.join(project_folder,'model')
os.makedirs(data_folder,exist_ok=True)
os.makedirs(model_folder,exist_ok=True)

In [148]:
sequence_length = 30
feature_count=63
samples_per_message=30
messages={'water':'I need water','help':'I need help','nurse':'Please call the nurse',
          'pain':'I am in pain','yes':'Yes','no':'No'}

In [149]:
mp_hands=mp.solutions.hands
mp_draw=mp.solutions.drawing_utils

In [150]:
hands=mp_hands.Hands(static_image_mode=False,max_num_hands=1,min_detection_confidence=0.5,
                     min_tracking_confidence=0.5)

In [151]:
video=cv2.VideoCapture(0)
while True:
    suc,img=video.read()
    img=cv2.flip(img,1)
    img1=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    result=hands.process(img1)
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(img,hand_landmarks,mp_hands.HAND_CONNECTIONS)
        cv2.imshow('Frame',img)
        if cv2.waitKey(1)&0XFF==ord('q'):
            break
video.release()
cv2.destroyAllWindows()

In [152]:
def get_hand_features(hand_landmarks):
    points=[]
    for landmark in hand_landmarks.landmark:
        points.append([landmark.x,landmark.y,landmark.z])
    points=np.array(points,dtype=np.float32)
    points=points-points[0]
    scale=np.linalg.norm(points[9])
    if scale>0:
        points=points/scale
    return points.flatten()

In [153]:
profile='user_1'
profile_data_folder=os.path.join(data_folder,profile)
os.makedirs(profile_data_folder,exist_ok=True)
for label in messages:
    label_folder=os.path.join(profile_data_folder,label)
    os.makedirs(label_folder,exist_ok=True)
print('Folders created')

Folders created


In [154]:
current_label='no'
print('Gesture:',current_label)
print('Message:',messages[current_label])

Gesture: no
Message: No


In [155]:
import time
save_folder=os.path.join(profile_data_folder, current_label)
os.makedirs(save_folder,exist_ok=True)
video=cv2.VideoCapture(0)
sequence = []
saved_count = len([file for file in os.listdir(save_folder)if file.endswith(".npy")])
print("Press S to save a sequence.")
print("Press Q to stop recording.")
while video.isOpened():
    success,img=video.read()
    if not success:
        print("Could not read webcam frame.")
        break
    img=cv2.flip(img, 1)
    img1=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img1)
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img,hand_landmarks,mp_hands.HAND_CONNECTIONS)
            features = get_hand_features(hand_landmarks)
            sequence.append(features)
            if len(sequence) > sequence_length:
                sequence.pop(0)
    else:
        sequence = []

    cv2.putText(img,f"Gesture: {current_label}",(10, 30),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0, 255, 0),2)
    cv2.putText(img,f"Frames: {len(sequence)}/{sequence_length}",(10, 65),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 255, 0),2)
    cv2.putText(img,f"Saved: {saved_count}",(10, 100),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 255, 0),2)
    cv2.imshow("Gesture Data Collection", img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("s"):
        if len(sequence) == sequence_length:
            filename = os.path.join(save_folder,f"sample_{saved_count:03d}.npy")
            np.save(filename, np.array(sequence))
            saved_count += 1
            sequence = []
            print("Saved:", filename)
            time.sleep(0.3)
        else:
            print("Keep your hand visible until 30 frames are collected.")
    elif key == ord("q"):
        break
video.release()
cv2.destroyAllWindows()
print("Total saved:", saved_count)

Press S to save a sequence.
Press Q to stop recording.
Total saved: 20


In [156]:
for label in messages:
    folder=os.path.join(profile_data_folder,label)
    files=[file for file in os.listdir(folder) if file.endswith(".npy")]
    print(label,":",len(files),"sequences")

water : 20 sequences
help : 20 sequences
nurse : 20 sequences
pain : 20 sequences
yes : 20 sequences
no : 20 sequences


In [157]:
x=[]
y=[]
for label in messages:
    folder = os.path.join(profile_data_folder, label)
    for filename in os.listdir(folder):
        if filename.endswith(".npy"):
            file_path = os.path.join(folder, filename)
            sequence = np.load(file_path)
            if sequence.shape == (sequence_length, feature_count):
                x.append(sequence)
                y.append(label)
x = np.array(x, dtype=np.float32)
y = np.array(y)
print("Data shape:", x.shape)
print("Labels shape:", y.shape)
print("Collected labels:", np.unique(y))

Data shape: (120, 30, 63)
Labels shape: (120,)
Collected labels: ['help' 'no' 'nurse' 'pain' 'water' 'yes']


In [158]:
labels=sorted(np.unique(y))
label_to_number={label:number for number,label in enumerate(labels)}
number_to_label={number:label for label,number in label_to_number.items()}
y_encoded=np.array([label_to_number[label] for label in y])
print('Label mapping:', label_to_number)
print('Encoded labels:', y_encoded[:10])

Label mapping: {np.str_('help'): 0, np.str_('no'): 1, np.str_('nurse'): 2, np.str_('pain'): 3, np.str_('water'): 4, np.str_('yes'): 5}
Encoded labels: [4 4 4 4 4 4 4 4 4 4]


In [159]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)
print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (96, 30, 63)
Testing data: (24, 30, 63)
Training labels: (96,)
Testing labels: (24,)


In [160]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
model = Sequential([Input(shape=(sequence_length, feature_count)),LSTM(64, return_sequences=True),Dropout(0.3),
                    LSTM(32),Dropout(0.3),Dense(32, activation="relu"),Dense(len(labels), activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 30, 64)         │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,438 (181.40 KB)

 Trainable params: 46,438 (181.40 KB)

 Non-trainable params: 0 (0.00 B)

In [161]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)
history = model.fit(x_train,y_train,epochs=60,batch_size=8,validation_split=0.2,callbacks=[early_stopping],verbose=1)

Epoch 1/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 135ms/step - accuracy: 0.1711 - loss: 1.7461 - val_accuracy: 0.0500 - val_loss: 1.8449
Epoch 2/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3553 - loss: 1.4963 - val_accuracy: 0.2000 - val_loss: 1.7697
Epoch 3/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5658 - loss: 1.2502 - val_accuracy: 0.2000 - val_loss: 1.7349
Epoch 4/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6053 - loss: 1.0658 - val_accuracy: 0.4000 - val_loss: 1.6449
Epoch 5/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7500 - loss: 0.8643 - val_accuracy: 0.6500 - val_loss: 1.4343
Epoch 6/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8816 - loss: 0.7024 - val_accuracy: 0.6500 - val_loss: 1.3500
Epoch 7/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9342 - loss: 0.5080 - val_accuracy: 0.7000 - val_loss: 1.1042
Epoch 8/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9868 - loss: 0.3856 - val_accuracy: 0.7000 -

In [162]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print('Test Loss:', round(test_loss, 4))
print('Test Accuracy:', round(test_accuracy*100,2), '%')

Test Loss: 0.1728
Test Accuracy: 91.67 %


In [163]:
model_path=os.path.join(model_folder,'gesture_model.h5')
model.save(model_path)
print('Location',model_path)

Location sign_free_project\model\gesture_model.h5


In [164]:
from collections import deque
# Load the saved model
model = tf.keras.models.load_model(model_path)
# Store the latest 30 frames
sequence = deque(maxlen=sequence_length)
video= cv2.VideoCapture(0)
display_message = "Show your hand"
confidence = 0.0
while video.isOpened():
    success,img = video.read()
    if not success:
        print("Cannot read webcam.")
        break
    img= cv2.flip(img, 1)
    img1 = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    results = hands.process(img1)
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(img,hand_landmarks,mp_hands.HAND_CONNECTIONS)
        features = get_hand_features(hand_landmarks)
        sequence.append(features)
        if len(sequence) == sequence_length:
            input_data = np.array(sequence, dtype=np.float32)
            input_data = np.expand_dims(input_data, axis=0)
            prediction = model.predict(input_data,verbose=0)[0]
            predicted_number = np.argmax(prediction)
            confidence = float(prediction[predicted_number])
            predicted_label = number_to_label[predicted_number]
            if confidence >= 0.75:
                display_message = messages[predicted_label]
            else:
                display_message = "Gesture unclear"
    else:
        sequence.clear()
        display_message = "Show your hand"
        confidence = 0.0
    cv2.putText(img,display_message,(10, 40),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0, 255, 0),2)
    cv2.putText(img,f"Confidence: {confidence:.2f}",(10, 80),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 255, 255),2)
    cv2.imshow("Sign-Free Communication System", img)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
video.release()
cv2.destroyAllWindows()